<a id="building-robust-llm-evaluation-pipelines"></a>
<div style="
  background: linear-gradient(145deg, #1a0b08, #2d1310);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #fff8f6;
  box-shadow: 0 6px 14px rgba(0,0,0,0.3);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #ff7b00, #ff0054, #9d0208);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>04 $\rightarrow$ Building Robust LLM Evaluation Pipelines</b>
  <br>
  <span style="color:#ffb5a7; font-size: 18px;">(Structural Diagnostics, Risk Taxonomy, and Enterprise Guardrails)</span>
</div>

---

# Table of Contents

1. [Overview of LLM Evaluations](#1-overview-of-llm-evaluations)
2. [Architectural Failure Points in LLM Systems](#2-architectural-failure-points-in-llm-systems)
   - 2.1 [Component-Level Evaluation](#21-component-level-evaluation)
   - 2.2 [Workflow-Level Evaluation](#22-workflow-level-evaluation)
   - 2.3 [Application-Level Evaluation](#23-application-level-evaluation)
3. [Case Study: RAG Pipeline Vulnerabilities](#3-case-study-rag-pipeline-vulnerabilities)
4. [Risk Categories in Evaluation](#4-risk-categories-in-evaluation)
   - 4.1 [Application Quality](#41-application-quality)
   - 4.2 [System Safety](#42-system-safety)
   - 4.3 [Operational Efficiency](#43-operational-efficiency)
   - 4.4 [Best Practices & Common Mistakes](#44-best-practices--common-mistakes)

---


<a id="1-overview-of-llm-evaluations"></a>
##
<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">1. Overview of LLM Evaluations</span>




<img src="../assets/nb_assets/nb0401.jpg" alt="nb0401.jpg" style="width:100%; max-width:700px; display:block; margin:auto;" />

### **Core Concepts**

Evaluation in the context of Large Language Models is the systematic and reliable measurement of models or LLM-based applications against established criteria. Operating an LLM application in production without rigorous evaluation pipelines guarantees unpredictable failures, unmonitored hallucinations, and degraded user experiences.

Evaluations are broadly categorized into two distinct domains:

- **Model Evaluation**
  - Benchmarking base or fine-tuned foundational models against standardized datasets (e.g., MMLU, HumanEval).
  - Typically performed by frontier AI laboratories to establish baseline capabilities of the model itself.

- **Application Evaluation**
  - Testing a custom application built on top of an LLM.
  - Evaluates how well the specific system—Including prompts, retrieval mechanisms, and external integrations—performs its designated business logic.

Engineering robust AI applications requires dedicating the majority of testing efforts toward **Application Evaluation**, as the base model is merely one component of the broader system. Because AI systems are non-deterministic and complex, a single evaluation metric or pipeline is insufficient. Production-grade LLM applications demand multiple, parallel evaluation pipelines.

### **The Two Fundamental Drivers for Multi-Pipeline Evaluation**

Production systems demand decoupled, dedicated evaluation pipelines operating concurrently due to two engineering realities:

1. **Multiple Failure Points Across Architectural Layers**
   - **Sub-components**
     - **Retrievers** — May return irrelevant or low-quality chunks due to poor embedding alignment or stale index data.
     - **Re-rankers** — Can misorder documents, burying critical context at lower positions where the generator ignores it.
     - **Parsers** — May fail to extract structured data from LLM outputs, breaking downstream tool calls or formatting.
   - **Workflow interactions**
     - **Context position bias** — LLMs disproportionately attend to tokens at the start (primacy) and end (recency) of the context window, ignoring middle chunks entirely (*Lost-in-the-Middle* effect).
     - **Attention decay** — As the context window fills with retrieved documents, generation quality degrades because the model's attention budget is spread too thin across competing chunks.
   - **System-level boundaries**
     - **Network latency** — API round-trips to embedding services, vector databases, and LLM endpoints add cumulative delay that compounds across multi-step pipelines.
     - **API expenditures** — Token costs scale with prompt length; bloated context windows from over-retrieval can multiply inference costs by 5–10×.

2. **Multiple Independent Risk Categories**
   - **Semantic correctness & groundedness**
     - Verifies that every claim in the generated output is directly supported by retrieved context — No fabrication, no extrapolation.
   - **Safety guardrails**
     - **Toxicity** — Detects and blocks hate speech, violent content, or dangerous instructions in generated responses.
     - **PII leaks** — Prevents exposure of social security numbers, credit card details, email addresses, or other personal data.
     - **Jailbreaks** — Defends against adversarial prompt injections that attempt to override system instructions.
   - **Operational throughput**
     - **Time-to-first-token (TTFT)** — Measures the delay before the user sees any streaming output; critical for perceived responsiveness.
     - **Financial cost** — Tracks per-request token consumption and third-party API spend to ensure unit economics remain viable at scale.

In [1]:
import random
random.seed(42)

# Stage reliabilities
stages = {
    "Parser": 0.98,
    "Retriever": 0.90,
    "Generator": 0.92,
    "Guardrail": 0.95,
}

runs = 100
successes = 0
failures = {stage: 0 for stage in stages}

# Simulate pipeline runs
for _ in range(runs):
    for stage, reliability in stages.items():
        if random.random() > reliability:
            failures[stage] += 1
            break
    else:
        successes += 1

# Results
print(f"Overall Success Rate: {successes}/{runs} ({successes}%)\n")
print("Failures per Stage:")
for stage, count in failures.items():
    print(f"  {stage:<10}: {count} failures")

Overall Success Rate: 72/100 (72%)

Failures per Stage:
  Parser    : 4 failures
  Retriever : 14 failures
  Generator : 7 failures
  Guardrail : 3 failures


<a id="2-architectural-failure-points-in-llm-systems"></a>
## 

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">2. Architectural Failure Points in LLM Systems</span>

### **Overview**

A primary driver for implementing multiple evaluation pipelines is the existence of numerous independent failure points within an AI architecture. Failures can occur in isolation or emerge from the interaction between otherwise perfectly functioning components.

Evaluations must be layered across three distinct architectural levels:

<img src="../assets/nb_assets/nb0402.jpg" alt="nb0402.jpg" style="width:100%; max-width:700px; display:block; margin:auto;" />

<a id="21-component-level-evaluation"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">2.1 Component-Level Evaluation</span>

Modern LLM applications are modular. Every individual component is a potential point of failure and requires its own isolated evaluation pipeline.

- **RAG Systems**
  - **Retriever** 
    - Fetches relevant documents from a vector store. 
      - Failures include returning semantically irrelevant chunks or missing the target document entirely.
  - **Reranker**
    - Re-scores and reorders retrieved chunks by query relevance.
      - Failures cause critical context to be buried at low-attention positions.
  - **Query Rewriter** 
    - Reformulates ambiguous user queries for better retrieval.
      - Failures produce distorted intent that derails the entire pipeline.
  - **Embedding Model**
    - Converts text to dense vector representations.
      - Drift or domain mismatch degrades retrieval precision silently.
  - **Vector Database**
    - Stores and indexes embeddings for similarity search.
      - Stale indices or configuration errors return outdated results.

- **Agentic Systems**
  - **Tool Selector**
    - Chooses which external API or function to invoke.
      - Incorrect selection triggers entirely wrong actions (e.g., calling `delete` instead of `read`).
  - **Output Parser** 
    - Extracts structured data (JSON, function calls) from raw LLM text.
      - Malformed parsing breaks downstream tool execution.
  - **Memory Module** 
    - Maintains conversation state across turns.
      - Failures cause the agent to forget prior context or repeat actions.
  - **Guardrails**
    - Enforces safety constraints and output validation.
      - Bypassed guardrails expose users to harmful or non-compliant content.

- **Generative Systems**
  - **System Prompt**
    - Defines the LLM's persona, constraints, and output format.
      - Poorly crafted prompts produce inconsistent or off-topic responses.
  - **Core LLM**
    - The foundation model that generates text.
      - Inherent biases, hallucination tendencies, and knowledge cutoffs are baseline failure modes.

<a id="22-workflow-level-evaluation"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">2.2 Workflow-Level Evaluation</span>

Components that perform flawlessly in isolation can still produce erroneous outputs when chained together. Workflow-level evaluation tests the integration, handoffs, and data flow between multiple interconnected stages.

- **Integration & Data Contracts**
  - **Schema & Type Validation**
    - Ensures data structures and serialization formats match downstream component expectations.
      - Failures cause silent field drops, parsing crashes, or type mismatch errors across step boundaries.
  - **Format Alignment**
    - Validates that text representations (HTML, Markdown, raw JSON) are preprocessed cleanly before reaching the generator.
      - Unstripped tags, noisy metadata, or malformed delimiters confuse prompt parsing and degrade output quality.
  - **State & Context Propagation**
    - Tracks the continuity of conversation history, tool outputs, and intermediate states across turns.
      - State drift or missing session context causes hallucinations and repeated reasoning loops.

- **Context & Attention Dynamics**
  - **Context Position Bias**
    - Optimizes the placement and ordering of retrieved evidence chunks within the model context window.
      - Key facts placed in middle positions suffer from attention decay (*Lost-in-the-Middle* effect) and get ignored.
  - **Attention Budget Saturation**
    - Manages total prompt token load and context density across chained calls.
      - Bloated context windows dilute attention mechanisms, leading to degraded reasoning and lost nuance.
  - **Distractor Interference**
    - Measures how well the generator ignores irrelevant or conflicting retrieved chunks.
      - High-similarity distractor chunks mislead the model into generating confident hallucinations.

- **Pipeline Failure Dynamics**
  - **Error Compounding (Cascade)**
    - Tracks how upstream degradation propagates and magnifies through sequential workflow steps.
      - Minor query rewriting flaws or suboptimal reranking cause complete failure in the final generation.
  - **Emergent Workflow Failures**
    - Identifies pipeline-level bugs where every individual unit-level component passes evaluation in isolation.
      - Subtle interface mismatches or timing differences cause systemic failure despite component-level success.

<a id="23-application-level-evaluation"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">2.3 Application-Level Evaluation</span>

Application-level evaluation assesses the system as a black box from the end user's perspective, verifying task fulfillment, operational constraints, and enterprise safety.

- **User Experience (UX) & Quality**
  - **Task Fulfillment**
    - Verifies that the end-to-end output completely and accurately satisfies the user's intent.
      - Failures produce incomplete answers, off-target guidance, or unaddressed sub-questions.
  - **Response Coherence & Style**
    - Evaluates linguistic fluency, persona consistency, clarity, and conciseness.
      - Failures produce rambling answers, inconsistent tone, or overly verbose boilerplate.
  - **Instruction Adherence**
    - Enforces strict compliance with requested output formats, tone constraints, and structural rules.
      - Failures include ignoring formatting constraints (e.g., failing to produce valid Markdown tables or JSON).

- **SLA & Operational Constraints**
  - **Latency & Responsiveness**
    - Measures Time-to-First-Token (TTFT) and end-to-end response duration against defined SLAs.
      - Excessive latency (e.g., 10s response against a <= 2s SLA) creates unacceptable user experience despite accurate answers.
  - **Token & Cost Viability**
    - Tracks token consumption, compute overhead, and per-query financial spend across APIs.
      - Unmonitored prompt bloating leads to unsustainable operational expenses at production scale.
  - **Throughput & Concurrency**
    - Evaluates system resilience, queue stability, and error rates under peak traffic loads.
      - Failures result in dropped requests, rate-limit throttling, and degraded server response times.

- **Enterprise Governance & Safety**
  - **Safety & Policy Guardrails**
    - Enforces strict screening against toxicity, hate speech, dangerous content, and policy violations.
      - Bypassed filters expose end-users to harmful material and create compliance liability.
  - **Data Privacy & Leakage**
    - Prevents inadvertent disclosure of Personally Identifiable Information (PII) or proprietary enterprise data.
      - Failures lead to severe regulatory fines and confidentiality breaches.
  - **Adversarial Robustness**
    - Validates defense resilience against direct/indirect prompt injections and jailbreak exploits.
      - Vulnerability exploits lead to unauthorized system prompt extraction or hijacked tool executions.

<a id="3-case-study-rag-pipeline-vulnerabilities"></a>
## 

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">3. Case Study: RAG Pipeline Vulnerabilities</span>

### **Overview**
To illustrate why multi-layered evaluation pipelines are necessary, consider a standard Retrieval-Augmented Generation (RAG) architecture used for an organizational knowledge base.

<img src="../assets/nb_assets/nb0403.png" alt="nb0403.jpg" style="width:100%; max-width:800px; display:block; margin:auto;" />

### **Analyzing the Failure Points**
In this pipeline, two critical components exist:
- **Retriever Evals:** Checks whether retrieved documents are relevant to the user query (*Context Relevance*).
- **Generator Evals:** Checks whether the generated output is strictly grounded in the retrieved context (*Faithfulness / Groundedness*).

---

### **The "Hidden" Workflow Failure Paradox**

Consider a scenario where the application receives the query:  
> 💬 *"What is the duration of the Machine Learning course?"*

- **Retrieval Step:** The Retriever is configured to fetch the top 5 documents ($K=5$) and returns chunks $D_1, D_2, D_3, D_4, D_5$.
- **Context Distribution:** The correct answer (*"8 weeks"*) is located in $D_5$. Documents $D_1$ through $D_4$ contain distracting info about a Python course lasting *"6 weeks"*.

| Evaluation Stage | Observed Behavior | Unit Result |
| :--- | :--- | :--- |
| **Component 1: Retriever** | Successfully retrieved the document containing the answer within top $K=5$. | **PASS** ✅ |
| **Component 2: Generator** | Prompted to prioritize top chunks ($D_1, D_2$), outputting: *"The duration is 6 weeks."* Faithfully followed prompt rules without hallucination. | **PASS** ✅ |
| **End-to-End System** | Delivered the **wrong answer** to the user. | **FAIL** ❌ |

#### **Mathematical Formulation of Context Position Bias**
When an LLM generator processes retrieved context chunks $C = \{c_1, c_2, \dots, c_K\}$, the attention probability weight assigned to chunk $c_i$ is non-uniform and depends heavily on its positional index $i$:

$$P(\text{Attention} \mid c_i) \propto \text{Primacy}(c_1, c_2) + \text{Recency}(c_K) - \text{Decay}(c_{\text{middle}})$$

> **Root Cause:** Because the target ground-truth fact was positioned at index $i = 5$ without an explicit re-ranking module, the generator prioritized information from higher-ranked chunks ($c_2$), resulting in an end-to-end failure despite acceptable individual component metrics.

In [ ]:
# Pipeline steps
def retriever(query):
    return ["Annual subscriptions cost $99.99."]

def generator(docs):
    return f"Based on docs: {docs[0]}"

# Run pipeline
query = "How do I get a refund?"
docs = retriever(query)
response = generator(docs)

# Evaluation checks
retriever_pass = len(docs) > 0
generator_pass = len(response) > 0
e2e_pass = "refund" in response.lower()

# Output results
print(f"Query:    {query}")
print(f"Output:   {response}\n")
print(f"Retriever Test:  {'PASS' if retriever_pass else 'FAIL'}")
print(f"Generator Test:  {'PASS' if generator_pass else 'FAIL'}")
print(f"End-to-End Test: {'PASS' if e2e_pass else 'FAIL'}")

Query:    How do I get a refund?
Output:   Based on docs: Annual subscriptions cost $99.99.

Retriever Test:  PASS
Generator Test:  PASS
End-to-End Test: FAIL


<a id="4-risk-categories-in-evaluation"></a>
## 

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">4. Risk Categories in Evaluation</span>

### **Overview**
Beyond mapping evaluations to architectural layers (Component, Workflow, Application), evaluations must also cover distinct **Risk Categories**.

Risk categories are divided into three primary pillars:

1. **Application Quality**
   - Accuracy — Is the answer factually correct?
   - Groundedness — Are all claims backed by retrieved context?
   - Relevance — Does the answer directly address the user's query?
   - Task completion — Did the system fulfill the user's end-to-end request?
2. **System Safety**
   - Toxicity — Blocks harmful, violent, or offensive content generation.
   - PII leaks — Prevents exposure of sensitive personal or financial data.
   - Jailbreaks — Defends against adversarial prompt injections that override system constraints.
3. **Operational Efficiency**
   - Latency — Measures total end-to-end response time in milliseconds.
   - Token consumption — Tracks prompt and completion token counts per request.
   - Error rates — Monitors HTTP failures, rate limits (429s), and schema validation errors.
   - Cost — Quantifies per-request spend across all third-party API calls.

<img src="../assets/nb_assets/nb0404.png" alt="nb0404.png" style="width:100%; max-width:600px; display:block; margin:auto;" />

<a id="41-application-quality"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.1 Application Quality</span>

These metrics determine whether the application accurately and effectively performs its primary business function across various architectural patterns.

### **Enterprise Risk Taxonomy Matrix**

| **Risk Domain** | **Risk Category** | **Description & Target Metric** |
| :--- | :--- | :--- |
| **Application Quality (General LLM)** | Correctness & Accuracy | Verifies factual truth against ground truth. |
| | Relevance & Directness | Measures query-to-answer semantic alignment. |
| | Completeness | Ensures all sub-questions are answered. |
| | Instruction Adherence | Validates structure, length, and format rules. |
| **Application Quality (RAG Specific)** | Context Relevance | Assesses signal-to-noise ratio in context. |
| | Groundedness / Faithfulness | Verifies claims are strictly backed by context. |
| | Citation Accuracy | Checks validity of inline context references. |
| **Application Quality (Agentic Workflows)** | Tool Selection Accuracy | Verifies correct API selection for tasks. |
| | Parameter Correctness | Checks valid schema formatting in tool calls. |
| | Trajectory Completion | Measures multi-step task completion success. |
| | Error State Recovery | Evaluates self-correction after API failures. |
| **Application Quality (Multi-Turn Chat)** | Context Retention | Tests state retention in multi-turn chats. |
| | Clarification Behavior | Verifies handling of ambiguous user prompts. |
| **Safety & Security** | Toxicity & Harmful Content | Detects offensive, violent, or dangerous text. |
| | PII & Data Leakage | Prevents exposure of sensitive personal data. |
| | Bias & Discrimination | Identifies demographic or political bias. |
| | Jailbreak Resistance | Measures robustness against prompt injections. |
| **Operational Telemetry** | End-to-End Latency | Measures total execution duration in milliseconds (ms). |
| | Time-To-First-Token (TTFT) | Measures delay before streaming output starts. |
| | Token Cost Efficiency | Tracks execution costs per 1,000 requests. |
| | Concurrency Failure Rate | Evaluates stability under concurrent load. |

<a id="42-system-safety"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.2 System Safety</span>

Safety evaluations ensure the application does not generate harmful, restricted, or biased content. These pipelines operate independently of quality checks.

- **Toxicity & Harmful Content**
  - Blocks requests and responses containing hate speech, violence, or dangerous instructions.
  - Uses classifier models (e.g., Perspective API, custom fine-tuned detectors) to score content toxicity before delivery.
- **Bias & Fairness**
  - Verifies equitable and impartial responses across demographic cohorts (gender, race, religion, age).
  - Tests for systematic skew in recommendations, hiring assessments, or content moderation decisions.
- **PII & Data Leakage**
  - Prevents exposure of sensitive credentials, payment details, or personal data (SSNs, credit cards, emails).
  - Scans both user inputs (to avoid logging sensitive data) and model outputs (to avoid surfacing memorized training data).
- **Jailbreak Resistance**
  - Assesses system defenses against adversarial prompt injection and system override attacks.
  - Tests with known jailbreak templates (DAN, role-play exploits, instruction leaking) to verify guardrail robustness.

<a id="43-operational-efficiency"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.3 Operational Efficiency</span>

Operational evaluations monitor infrastructure health, performance budgets, and resource utilization in production.

- **Latency (TTFT & E2E)**
  - **Time-to-First-Token (TTFT)** — Measures the delay (ms) before the user sees any streaming output; directly impacts perceived responsiveness.
  - **End-to-End (E2E) latency** — Total round-trip duration from user query submission to final response delivery, including all retrieval, reranking, and generation steps.
- **Cost per Request**
  - Quantifies input/output token consumption and third-party API spend per inference call.
  - Bloated context windows from over-retrieval can multiply costs by 5–10× without proportional quality gains.
- **Token Efficiency**
  - Minimizes redundant prompt tokens (duplicate context, verbose system instructions) while preserving generation quality.
  - Tracks the ratio of useful output tokens to total tokens consumed per request.
- **Failure/Error Rate**
  - **HTTP timeouts** — Requests that exceed maximum wait time and return no response.
  - **Rate limits (429)** — API throttling errors from exceeding provider quotas during high-concurrency loads.
  - **Schema decoding errors** — Malformed JSON or unexpected output structure from the LLM that breaks downstream parsing.

---

### **Implementation Framework: Production Multi-Pipeline Evaluator**

The following Python script simulates an evaluation engine that executes parallel pipelines across three independent domains: **Application Quality (Faithfulness)**, **Safety (PII Leakage)**, and **Operations (Latency SLA)**.

In [3]:
# Multi-Dimensional Risk Taxonomy Evaluator
import re

def evaluate_response(response, context, latency_ms):
    # 1. Quality Check (Word Overlap Faithfulness)
    resp_words = set(re.findall(r'\b\w{4,}\b', response.lower()))
    ctx_words = set(re.findall(r'\b\w{4,}\b', context.lower()))
    faithfulness = len(resp_words & ctx_words) / len(resp_words) if resp_words else 0.0
    
    # 2. Safety Check (PII - Social Security Number)
    has_pii = bool(re.search(r'\b\d{3}-\d{2}-\d{4}\b', response))
    
    # 3. Operational Check (SLA Latency <= 1000ms)
    latency_ok = latency_ms <= 1000

    return {
        "quality_pass": faithfulness >= 0.5,
        "safety_pass": not has_pii,
        "operational_pass": latency_ok,
    }

scenarios = [
    {"name": "Clean Run",    "response": "Refunds are processed in 5 business days.", "ctx": "Refunds take 5 business days.", "lat": 450},
    {"name": "PII Leak",     "response": "User SSN is 000-12-3456.",                  "ctx": "User data stored safely.",        "lat": 300},
    {"name": "High Latency", "response": "Refunds take 5 days.",                     "ctx": "Refunds take 5 days.",            "lat": 2500},
]

for sc in scenarios:
    res = evaluate_response(sc["response"], sc["ctx"], sc["lat"])
    overall = all(res.values())
    status = "PASS" if overall else "FAIL"
    q_str = 'PASS' if res['quality_pass'] else 'FAIL'
    s_str = 'PASS' if res['safety_pass'] else 'FAIL'
    o_str = 'PASS' if res['operational_pass'] else 'FAIL'
    print(f"Scenario: {sc['name']:<14} -> [{status}] (Quality: {q_str}, Safety: {s_str}, SLA Latency: {o_str})")

Scenario: Clean Run     -> [PASS] (Quality: PASS, Safety: PASS, SLA Latency: PASS)
Scenario: PII Leak      -> [FAIL] (Quality: FAIL, Safety: FAIL, SLA Latency: PASS)
Scenario: High Latency  -> [FAIL] (Quality: PASS, Safety: PASS, SLA Latency: FAIL)


<a id="44-best-practices--common-mistakes"></a>
### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.4 Best Practices & Common Mistakes</span>

#### **Best Practices**

- **Decouple and Run Safety Checks Asynchronously (In Parallel)**
  - Run security and safety filters (detecting toxic content or PII) in parallel with quality checks.
  - Asynchronous execution prevents safety guardrails from adding extra waiting time to user responses.

- **Set Strict SLA (Service Level Agreement) Thresholds & Alerts**
  - **SLA (Service Level Agreement)** defines the agreed-upon speed, uptime, and reliability boundaries for a system.
  - Monitor **TTFT (Time-to-First-Token)** — the initial delay in milliseconds before the user sees the first streamed token (e.g., alert if $\text{TTFT} > 1{,}000\text{ ms}$).
  - Monitor **E2E (End-to-End) Latency** — total round-trip duration from user query submission to complete response delivery.

- **Screen for PII (Personally Identifiable Information)**
  - **PII (Personally Identifiable Information)** includes sensitive user data like Social Security Numbers (SSNs), credit cards, home addresses, or phone numbers.
  - Implement automated regex and named-entity recognition (NER) redaction filters before and after generation.

- **Use Dedicated, Specialized Evaluation Prompts for Each Risk Category**
  - Evaluate accuracy, safety, and brand tone with separate, single-purpose evaluation prompts.
  - Dedicated prompts prevent the LLM judge from suffering attention decay across multiple competing criteria.

---

#### **Common Mistakes**

- **Treating a Correct Answer as a Success When It Violates SLA Limits**
  - Assuming a response is acceptable simply because it is factually accurate, even if it took 10 seconds to generate.
  - If the operational SLA is $\le 2\text{ s}$, a 10-second response is an operational failure that ruins user experience.

- **Combining All Evaluation Metrics into a Single Massive Prompt**
  - Asking a single LLM judge to simultaneously grade factual correctness, tone, PII leaks, and formatting rules.
  - Overloaded evaluation prompts dilute model attention, leading to missed errors and unreliable scoring.

- **Overlooking Compounding Token Costs in Multi-Step Workflows**
  - Optimizing solely for accuracy benchmarks while ignoring the token consumption of multi-turn agent loops and oversized retrieval chunks.
  - An application that scores 98% on benchmarks but costs $0.50 per query will fail business unit economics at production scale.

- **Neglecting Intermediate Component Failures (Silent Data Corruption)**
  - Testing only the final output string and assuming passing text means the retrieval and parser stages worked correctly.
  - Unchecked retriever errors often produce subtle hallucinations that go undetected without isolated component diagnostics.

---

### Key Takeaways
- Multi-pipeline evaluation categorizes risks into **Application Quality**, **Safety & Security**, and **Operational Telemetry**.
- Enterprise production gates require simultaneous pass status across all three dimensions before code deployment.